In [ ]:
import pybliometrics
from pybliometrics.scopus import ScopusSearch
from pybliometrics.scopus.utils import config
from pybliometrics.scopus import AuthorRetrieval
import sys
import pandas as pd
import time

s = ScopusSearch("AU-ID(56344636600)")
my_papers = s.results
my_papers = [[x.title, x.eid, x.doi] for x in my_papers if x.eid not in ["2-s2.0-84906819575", "2-s2.0-84976641685"]]
my_papers

In [ ]:
dfp = pd.DataFrame(my_papers, columns=["Paper title", "Paper eid", "Paper doi"])
dfp.sort_values(["Paper title"], ignore_index=True)

In [ ]:
acc = []
for paper in my_papers:
    q = f"REF({paper[1]})"
    s = ScopusSearch(q, refresh=0)
    print(s)
    res = s.results
    if res is not None and len(res) > 0:
        for x in res:
            acc = acc + [paper + [x.title, x.eid, x.doi]]
    else:
        acc = acc + [paper + ["", "", ""]]

In [ ]:
df = pd.DataFrame(acc, columns=["Paper title", "Paper eid", "Paper doi", "Citing paper title", "Citing paper eid", "Citing paper doi"])
df.to_csv("data/scopus-citations-{}.csv".format(time.time()), index=False)
df

In [ ]:
import numpy as np
df = df.replace("", np.nan)
df["Citing paper title"].isna()

Total citations

In [ ]:
s.count().sum()

Citation details

In [ ]:
s = df.groupby(["Paper title"])["Citing paper title"]
s.count().sort_values(ascending=False).to_csv("data/scopus-summary-{}.csv".format(time.time()), index=False)